# 03 - Feature engineering (part 1)

Adds the features that need no text parsing, then gives an overview of the resulting data.

| Group | Features | Built from |
|---|---|---|
| Time (UTC) | `utc_date`, `utc_month`, `utc_iso_week`, `utc_weekday`, `utc_weekday_name`, `utc_is_weekend`, `utc_hour` | `epoch_seconds` |
| Parties | `party_pair` | `ordering_type`, `beneficiary_type` |
| Amount | `usd_amount_log10`, `usd_amount_magnitude`, `usd_amount_is_whole` | `usd_amount` |

**Input:** `data/processed/transactions_clean.parquet` (run `01_preprocessing.ipynb` first).  
**Output:** `data/processed/transactions_features.parquet` = every clean column, untouched, plus the new columns.

Geography, routing, account, token and anomaly-signature features are added later, once the text-column formats are confirmed in `02` (EDA).

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"Project root not found from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import features
from src import settings as S

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)
print("Project root:", ROOT)

Project root: c:\Universiti Malaya (Extra)\DATATHON\liqi-datathon


## 1. Build the features

Processed in batches, so memory use stays low. The original columns are passed through unchanged.

In [2]:
features.run()

  rows processed: 10,000,000 / 10,000,000
Added 19 features to 21 clean columns = 40 columns, 10,000,000 rows
Missing values in new features: none
Saved -> data\processed\transactions_features.parquet
Feature dictionary -> reports\profiling\feature_dictionary.csv


What each new column means:

In [3]:
pd.read_csv(features.FEATURE_DICTIONARY_CSV)

,feature,group,type,source,description
0,utc_date,time,date32[day],epoch_seconds,Calendar date of the payment in UTC
1,utc_month,time,int8,epoch_seconds,Month number 1-12 in UTC
2,utc_iso_week,time,int8,epoch_seconds,ISO 8601 week number in UTC (weeks start on Monday)
3,utc_weekday,time,int8,epoch_seconds,Day of week in UTC: 0 = Monday ... 6 = Sunday
4,utc_weekday_name,time,"dictionary<values=string, indices=int32, ordered=0>",epoch_seconds,Day of week in UTC as Mon ... Sun
5,utc_is_weekend,time,bool,epoch_seconds,True when the UTC day is Saturday or Sunday
6,utc_hour,time,int8,epoch_seconds,Hour of day 0-23 in UTC
7,party_pair,parties,"dictionary<values=string, indices=int32, ordered=0>",ordering_type + beneficiary_type,"Sender type to receiver type: B2B, B2C, C2B or C2C (B = business, C = consumer)"
8,usd_amount_log10,amount,double,usd_amount,log10 of usd_amount (missing if the amount is not above 0)
9,usd_amount_magnitude,amount,int8,usd_amount,"Order of magnitude: largest k with 10^k <= usd_amount (0.01 -> -2, 999.99 -> 2, 1000 -> 3)"


**Notes on interpretation**

- All time features are in **UTC**, not the local time of the sender or receiver. A payment at 09:00 in Kuala Lumpur shows as `utc_hour = 1`. Local-time features come later, with the geography group.
- `utc_is_weekend` means Saturday or Sunday in UTC. Around midnight UTC this can differ from the local day, and some countries have a different weekend.
- `usd_amount` is already converted to USD, so `usd_amount_is_whole` only shows a round amount reliably when the payment currency is USD. For other currencies a round original amount usually stops being round after conversion.

## 2. Overview of the data after the process

### 2.1 Size

In [4]:
pf = pq.ParquetFile(features.FEATURES_PARQUET)
schema = pf.schema_arrow
new_cols = [f[0] for f in features.FEATURES]
original_cols = [c for c in schema.names if c not in new_cols]

print(f"Rows            : {pf.metadata.num_rows:,}")
print(f"Columns         : {len(schema.names)}  ({len(original_cols)} from the clean file + {len(new_cols)} new)")
print(f"Clean file      : {S.CLEAN_PARQUET.stat().st_size / 1e6:,.1f} MB")
print(f"Features file   : {features.FEATURES_PARQUET.stat().st_size / 1e6:,.1f} MB")

Rows            : 10,000,000
Columns         : 40  (21 from the clean file + 19 new)
Clean file      : 1,554.4 MB
Features file   : 1,705.9 MB


### 2.2 All columns

In [5]:
pd.DataFrame({
    "column": schema.names,
    "type": [str(t) for t in schema.types],
    "origin": ["new feature" if c in new_cols else "clean file" for c in schema.names],
})

,column,type,origin
0,epoch_seconds,"timestamp[us, tz=UTC]",clean file
1,currency_code,string,clean file
2,usd_amount,double,clean file
3,ordering_bic8,string,clean file
4,beneficiary_bic8,string,clean file
5,bic8_path,string,clean file
6,total_duration,int64,clean file
7,continuity_qflag,bool,clean file
8,currency_conversion_count,int64,clean file
9,participant_count,int64,clean file


### 2.3 First 10 rows

In [6]:
next(pf.iter_batches(batch_size=10)).to_pandas()

,epoch_seconds,currency_code,usd_amount,ordering_bic8,beneficiary_bic8,bic8_path,total_duration,continuity_qflag,currency_conversion_count,participant_count,transaction_type_code,ordering_account,beneficiary_account,ordering_details,ordering_tokens,beneficiary_details,beneficiary_tokens,ordering_type,beneficiary_type,anomaly_type,is_anomaly,utc_date,utc_month,utc_iso_week,utc_weekday,utc_weekday_name,utc_is_weekend,utc_hour,party_pair,usd_amount_log10,usd_amount_magnitude,usd_amount_is_whole,sender_country,receiver_country,is_domestic,country_corridor,path_entries,path_gaps,path_first_differs,path_last_differs
0,2023-05-29 12:29:04+00:00,USD,500.00,0621US30,0219US30,0621US30;0219US30,4,True,0,2,Direct,_FAKE_00.-0/0092Y9262,_FAKE_0009559K28N,/00.-0/0092Y9262\nSCHULTZ PLC 38626 CHRISTENSEN GARDE\nN 30410 US\n8530276,/<chr:99.-9/9999A9999>\n<line_break:35><business:7> <company_keyword:3> <other_words:2> <street:...,/0009559K28N\nS MRDHAA 66 SIRAJROAD 2711 GAAIBAAN\nDHAA BD,/<chr:9999999A99A>\n<line_break:35><chr:A> <last_name:6> <chr:99> <street:9> <post_code:BD:4> <c...,B,C,NaN,0,2023-05-29,5,22,0,Mon,False,12,B2C,2.698970,2,True,US,US,True,US->US,2,0,False,False
1,2023-06-13 12:25:45+00:00,TRY,1390.34,0024TRI0,0380CNB0,0024TRI0;0380CNB0,396,True,1,2,Direct,_FAKE_TR53002400000000890222,_FAKE_0000000008A79136,/TR53002400000000890222\nCALIK 1162 SERAFEDDIN VILLE 63439 T\nR,/<iban:22>\n<line_break:35><business:5> <chr:9999> <street:16> <post_code:TR:5> <country_code:TR>,//0000000008A79136\nGUI LAN XIE DZUO TAO LU CN,/<separator:/><chr:9999999999A99999>\n<first_name:8> <other_words:13> <chr:AAAA > <street:7> <co...,B,C,NaN,0,2023-06-13,6,24,1,Tue,False,12,B2C,3.143121,3,False,TR,CN,False,TR->CN,2,0,False,False
2,2023-07-04 17:38:32+00:00,GBP,3972.37,0007JEH0,0295GB20,0007JEH0;0295GB20,3,True,0,2,Direct,_FAKE_00 000 000B0623 133W,_FAKE_GB30029500000000623158,"/00 000 000B0623 133W\n1/GORDON, SMITH AND MARTIN\n2/BROWN PORT 536\n3/JE/ALLANSHIRE",/<chr:99 999 999A9999 999A>\n<other_words:26>\n2/<street:10> <chr:999>\n3/<country_code:JE>/<cit...,/GB30029500000000623158\nW GUI XIANG YZUO XIANG GANG JIE\nHK,/<iban:22>\n<chr:A> <last_name:10> <chr:AAAA > <street:15> <country_code:HK>,B,C,NaN,0,2023-07-04,7,27,1,Tue,False,17,B2C,3.599050,3,False,JE,GB,False,JE->GB,2,0,False,False
3,2023-05-19 03:06:09+00:00,SGD,1920.69,0203SGS0,0037DEF0,0203SGS0;0405US30;0037DEF0,4341,True,2,3,Indirect,_FAKE_00000 F00/08/57/6/40,_FAKE_DE49003700000000292310,/00000 F00/08/57/6/40\nPERUM WASKITA JL. DIPENOGORO 2 2259\n7 SG,/<chr:99999 A99/99/99/9/99>\n<line_break:35><business:5> <company_keyword:7> <street:14> <chr:9>...,/DE49003700000000292310\nM ALBA 80036 CIRCUNVALACION ANDORRA\nSAN TIMOTEO DE LA MONTANA MX,/<iban:22>\n<other_words:6> <last_name:4> <chr:99999> <street:22> <city:MX:25> <country_code:MX>,B,C,NaN,0,2023-05-19,5,20,4,Fri,False,3,B2C,3.283457,3,False,SG,DE,False,SG->DE,3,0,False,False
4,2023-06-05 21:32:56+00:00,EUR,22130.99,0017DEF0,0087ZAJ0,0017DEF0;0087ZAJ0,20,True,1,2,Direct,_FAKE_DE13001700000000291498,_FAKE_ 0000000.00099.3989,/DE13001700000000291498\nKREUSEL 1-9 ANA-BIRNBAUM-RING FINST\nERWALDE DE,/<iban:22>\n<line_break:35><business:7> <chr:9-9> <street:17> <city:DE:12> <country_code:DE>,/ 0000000.00099.3989\nMHWRSZN YRN KHWDRW SHHRSTN `LY,/<chr: 9999999.99999.9999>\n<business:18> <city:AF:11>,B,B,NaN,0,2023-06-05,6,23,0,Mon,False,21,B2B,4.345001,4,False,DE,ZA,False,DE->ZA,2,0,False,False
5,2023-06-12 19:21:22+00:00,EUR,349.95,0001MQM0,0009TNT0,0001MQM0;0009TNT0,1588,True,1,2,Direct,_FAKE_00E69 0N 873,_FAKE_TN11000900000000886688,/00E69 0N 873\nLE,/<chr:99A99 9A 999>\n<business:2>,//TN11000900000000886688\nBIN BAO ZHONG QING JIE VZUO 971\n012 MA AN SHAN XIAN CN,/<separator:/><iban:22>\n<line_break:35><first_name:4> <last_name:4> <street:15> <chr:AAAA > <po...,B,C,NaN,0,2023-06-12,6,24,0,Mon,False,19,B2C,2.544006,2,False,MQ,TN,False,MQ->TN,2,0,False,False
6,2023-06-13 15:03:59+00:00,EUR,1447.90,0412FRP0,0031TWT0,0412FRP0;0023FIH0;0035LUL0

### 2.4 New features: missing values and range

Only the new feature columns (plus the label) are loaded here, so this stays light on memory.

In [7]:
extra = [c for c in ["anomaly_type", "is_anomaly"] if c in schema.names]
df = pq.read_table(features.FEATURES_PARQUET, columns=new_cols + extra).to_pandas(date_as_object=False)
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB in memory")

rows = []
for c in new_cols:
    s = df[c]
    ordered = s.dtype.kind in "iufM"  # numbers and dates have a meaningful min/max
    rows.append({
        "feature": c,
        "dtype": str(s.dtype),
        "missing": int(s.isna().sum()),
        "distinct": int(s.nunique()),
        "min": s.min() if ordered else "",
        "max": s.max() if ordered else "",
    })
pd.DataFrame(rows)

10,000,000 rows x 21 columns, 381.5 MB in memory


,feature,dtype,missing,distinct,min,max
0,utc_date,datetime64[ms],0,93,2023-04-30 00:00:00,2023-07-31 00:00:00
1,utc_month,int8,0,4,4,7
2,utc_iso_week,int8,0,15,17,31
3,utc_weekday,int8,0,7,0,6
4,utc_weekday_name,category,0,7,,
5,utc_is_weekend,bool,0,2,,
6,utc_hour,int8,0,24,0,23
7,party_pair,category,0,4,,
8,usd_amount_log10,float64,0,3464639,-2.0,10.194906
9,usd_amount_magnitude,int8,0,13,-2,10


### 2.5 Time coverage

In [8]:
per_day = df.groupby("utc_date").size()
print(f"First day   : {per_day.index.min().date()}")
print(f"Last day    : {per_day.index.max().date()}")
print(f"Days covered: {len(per_day)} (calendar span {(per_day.index.max() - per_day.index.min()).days + 1} days)")
print(f"Payments per day: min {per_day.min():,} | median {int(per_day.median()):,} | max {per_day.max():,}")
print("Note: the first and last day can be partial.")

First day   : 2023-04-30
Last day    : 2023-07-31
Days covered: 93 (calendar span 93 days)
Payments per day: min 2,311 | median 109,549 | max 220,212
Note: the first and last day can be partial.


### 2.6 Distribution of each new feature

In [9]:
def counts(col, sort_index=True):
    t = df[col].value_counts(dropna=False)
    t = t.sort_index() if sort_index else t
    return pd.DataFrame({"rows": t, "pct": (t / len(df) * 100).round(4)})

display(counts("party_pair"))
display(counts("utc_month"))
display(counts("utc_weekday_name"))
display(counts("utc_is_weekend"))

,rows,pct
party_pair,,
B2B,2270945,22.7094
B2C,5603692,56.0369
C2B,610605,6.1060
C2C,1514758,15.1476


,rows,pct
utc_month,,
4,6161,0.0616
5,3044400,30.4440
6,4050160,40.5016
7,2899279,28.9928


,rows,pct
utc_weekday_name,,
Mon,1833685,18.3368
Tue,1708480,17.0848
Wed,1704615,17.0461
Thu,1801042,18.0104
Fri,1833702,18.3370
Sat,434935,4.3493
Sun,683541,6.8354


,rows,pct
utc_is_weekend,,
False,8881524,88.8152
True,1118476,11.1848


In [10]:
display(counts("utc_hour"))
display(counts("utc_iso_week"))

,rows,pct
utc_hour,,
0,308404,3.0840
1,258682,2.5868
2,233387,2.3339
3,218010,2.1801
4,224509,2.2451
5,308543,3.0854
6,488784,4.8878
7,502613,5.0261
8,466923,4.6692


,rows,pct
utc_iso_week,,
17,6161,0.0616
18,745342,7.4534
19,500252,5.0025
20,503176,5.0318
21,849087,8.4909
22,849665,8.4966
23,750832,7.5083
24,854063,8.5406
25,1198021,11.9802


In [11]:
display(counts("usd_amount_magnitude"))
display(counts("usd_amount_is_whole"))
df["usd_amount_log10"].describe().to_frame().T

,rows,pct
usd_amount_magnitude,,
-2,1026,0.0103
-1,10863,0.1086
0,122712,1.2271
1,715250,7.1525
2,2223897,22.2390
3,3442986,34.4299
4,2514618,25.1462
5,838507,8.3851
6,122555,1.2256


,rows,pct
usd_amount_is_whole,,
False,9259963,92.5996
True,740037,7.4004


,count,mean,std,min,25%,50%,75%,max
usd_amount_log10,10000000.0,3.54337,1.122868,-2.0,2.802149,3.563706,4.300157,10.194906


### 2.7 Quick check against the brief

The brief defines a romance scam as consumer to consumer, and a CEO scam as a payment made by a business. If `party_pair` is built correctly, the labelled rows should show that pattern.

In [12]:
if "anomaly_type" in df.columns:
    display(pd.crosstab(df["anomaly_type"], df["party_pair"], margins=True, margins_name="Total"))
else:
    print("anomaly_type is not in the data - skipped")

party_pair,B2B,B2C,C2B,C2C,Total
anomaly_type,,,,,
ACCOUNT_REPLACEMENT_SCAM,138,370,34,107,649
CEO_SCAM,262,683,0,0,945
CUSTOMER_RECALL,63,183,22,49,317
DUPLICATE_PAYMENT,152,393,21,113,679
FAT_FINGER_MISTAKE,445,1098,101,296,1940
ROMANCE_SCAM,0,0,665,1811,2476
Total,1060,2727,843,2376,7006
